# GKBA module — example notebook

Demonstrates how to use the `GKBA` module to run a simulation and inspect observables.
We use the **WBL case** (fastest to run) with two precessing spins as a concrete example.

The setup:
- 2-site chain, 2 leads
- Local moments precessing at angle θ=20° with period T=5
- Lead coupling turned on via a sigmoid envelope
- Observables: charge current, spin current, site spin density

In [ ]:
using DifferentialEquations
using Plots

import GKBA: init_gkba_wbl, eom_gkba_wbl!, compute_observables!,
             PrecSpin, update!, build_hs, unpack!

## Parameters

In [ ]:
nx, ny  = 2, 1
nk      = 20        # Ozaki poles: n_oz = nk * nσ
γ       = 1.0
γso     = 0.0
γc      = 1.0
j_sd    = 0.1
Temp    = 300.0

t_0     = 0.0
t_step  = 0.1
t_end   = 60.0

theta_1 = 20.0      # precession cone angle (degrees)
phi_1   = 0.0
period  = 5.0

stepp(t; ti=3, to=25) = 1 / (1 + exp(-(t - to) / ti))   # coupling envelope

## Initialise

In [ ]:
pr_spins = [PrecSpin(i; theta_zero=theta_1, phi_zero=phi_1, T=period) for i in 1:nx]

vm_init = zeros(Float64, nx*ny, 3)
for ps in pr_spins
    update!(ps, 0.0)
    vm_init[ps.i, :] .= ps.s
end

dv, ov = init_gkba_wbl(; nx, ny, nk, γ, γso, γc, j_sd, Temp, vm_i1x=vm_init)
dv.s_α .= stepp(t_0)

println("System size:   ", size(dv.Gls_ij))
println("Ozaki poles:   ", length(dv.ξ_k))
println("State vector:  ", length(dv.rkvec), " complex DOF")

## Time evolution

In [ ]:
prob       = ODEProblem(eom_gkba_wbl!, dv.rkvec, (t_0, t_end), dv)
integrator = init(prob, RK4(); dt=t_step, save_everystep=false, adaptive=true, dense=false)

ts      = Float64[]
curr    = Vector{Float64}[]
scurr   = Vector{Float64}[]
sden    = Vector{Float64}[]
cspins  = Vector{Float64}[]

for t in t_0:t_step:t_end-t_step
    tt = round(t + t_step, digits=2)

    step!(integrator, t_step, true)
    unpack!(dv, integrator.u)
    compute_observables!(ov, dv)

    for ps in pr_spins
        update!(ps, tt)
        dv.vm_i1x[ps.i, :] .= ps.s
    end
    dv.hs_ij = build_hs(dv.vm_i1x, nx, ny, γ, γso, j_sd)
    dv.s_α  .= stepp(tt)

    push!(ts,     tt)
    push!(curr,   copy(real(ov.curr_α)))
    push!(scurr,  copy(real(ov.scurr_xα[:, 1])))   # spin current lead 1 (x,y,z)
    push!(sden,   copy(real(ov.sden_i1x[1, :])))
    push!(cspins, copy(real(dv.vm_i1x[1, :])))
end

println("Done.")

## Charge current

In [ ]:
curr_mat = hcat(curr...)'   # (nt, nα)

plot(ts, curr_mat[:, 1], label="Lead 1", xlabel="t", ylabel="J (charge)", lw=1.5)
plot!(ts, curr_mat[:, 2], label="Lead 2", lw=1.5)
plot!(ts, curr_mat[:, 1] .+ curr_mat[:, 2], label="Sum", lw=1, ls=:dash, color=:black)

## Spin current (lead 1)

In [ ]:
sc_mat = hcat(scurr...)'   # (nt, 3)

plot(ts, sc_mat[:, 1], label="x", xlabel="t", ylabel="J (spin, lead 1)", lw=1.5)
plot!(ts, sc_mat[:, 2], label="y", lw=1.5)
plot!(ts, sc_mat[:, 3], label="z", lw=1.5)

## Non-equilibrium spin density — site 1

In [ ]:
sden_mat   = hcat(sden...)'    # (nt, 3)
cspins_mat = hcat(cspins...)'  # (nt, 3)

plot(ts, sden_mat[:, 1],   label="⟨Sˣ⟩", xlabel="t", ylabel="Spin density (site 1)", lw=1.5)
plot!(ts, sden_mat[:, 2],  label="⟨Sʸ⟩", lw=1.5)
plot!(ts, sden_mat[:, 3],  label="⟨Sᶻ⟩", lw=1.5)
plot!(ts, cspins_mat[:, 1], label="mˣ (classical)", lw=1, ls=:dash)
plot!(ts, cspins_mat[:, 3], label="mᶻ (classical)", lw=1, ls=:dash)

## Quick checks

In [ ]:
# G^< should remain anti-Hermitian throughout
G = dv.Gls_ij
println("max |G + G†| = ", maximum(abs.(G .+ G')))

# Charge density (should be ~1 electron per site at half-filling)
compute_observables!(ov, dv)
println("Charge density: ", round.(real(ov.cden_i), digits=4))